In [4]:
import pandas as pd
import numpy as np
import json

# 1. LOAD DATASET FROM PATH 
file_path = r'C:\Users\aryan\Downloads\assig\BrightChamps_FDA_Case_Dataset.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset successfully loaded!\n")
except Exception as e:
    print(f"Error loading file from path '{file_path}': {e}")
    # Fallback to local file if needed
    df = pd.read_csv('BrightChamps_FDA_Case_Dataset.csv')


Dataset successfully loaded!



In [6]:
# 2. DATA PREPROCESSING & STANDARDIZATION
df.columns = df.columns.str.strip().str.lower()

# Standardize string fields
string_cols = ['lead_source', 'geography', 'parent_timezone', 'rep_assigned', 'rep_shift', 'demo_joined', 'demo_completed', 'converted']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Clean Y/N flags
df['demo_joined_flag'] = df['demo_joined'].str.upper() == 'Y'
df['demo_completed_flag'] = df['demo_completed'].str.upper() == 'Y'
df['converted_flag'] = df['converted'].str.upper() == 'Y'

# Corrected column name mapping based on dataset: 'demo_scheduled_at'
scheduled_col = 'demo_scheduled_at' if 'demo_scheduled_at' in df.columns else 'demo_scheduled_timestamp'
created_col = 'created_at' if 'created_at' in df.columns else 'created_timestamp'

# Check if demo was scheduled
df['demo_scheduled_flag'] = df[scheduled_col].notna() & (df[scheduled_col].astype(str).str.strip() != '') & (df[scheduled_col].astype(str).str.lower() != 'nan')

# Convert timestamps to datetime objects
for col in [created_col, scheduled_col]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

In [7]:
# 3. FUNNEL STAGE METRICS CALCULATION
total_leads = len(df)
demos_scheduled = df['demo_scheduled_flag'].sum()
demos_joined = df['demo_joined_flag'].sum()
demos_completed = df['demo_completed_flag'].sum()
converted_customers = df['converted_flag'].sum()

# Monthly scaling factor (dataset spans 2 months)
num_months = 2.0
monthly_leads = total_leads / num_months
monthly_scheduled = demos_scheduled / num_months
monthly_joined = demos_joined / num_months
monthly_no_shows = (demos_scheduled - demos_joined) / num_months

# Business & Unit Economics Parameters
REV_PER_CONVERTED = 60000  # ₹60,000 per customer
CAC_PER_LEAD = 900          # ₹900 per lead

# Downstream Conversion Rates
completion_rate = demos_completed / demos_joined if demos_joined > 0 else 0
closing_rate = converted_customers / demos_completed if demos_completed > 0 else 0
overall_joined_to_paid = completion_rate * closing_rate

# Expected Gross Value per Joined Demo
expected_value_per_joined_demo = overall_joined_to_paid * REV_PER_CONVERTED

# Financial Leak Sizing (Monthly)
monthly_unrealized_revenue = monthly_no_shows * expected_value_per_joined_demo
monthly_sunk_cac = monthly_no_shows * CAC_PER_LEAD

In [8]:
# 4. ROOT CAUSE ANALYSIS: TIMEZONE VS REP SHIFT MISMATCH
# Shift expectation mapping based on parent timezone
tz_to_expected_shift = {
    'America/New_York': 'US_SHIFT',
    'America/Los_Angeles': 'US_SHIFT',
    'Europe/London': 'IST_SHIFT',    # or EMEA/SEA depending on team allocation
    'Asia/Dubai': 'IST_SHIFT',
    'Asia/Riyadh': 'IST_SHIFT',
    'Asia/Singapore': 'IST_SHIFT',
    'Asia/Ho_Chi_Minh': 'SEA_SHIFT',
    'Australia/Sydney': 'US_SHIFT'
}

df['expected_shift'] = df['parent_timezone'].map(tz_to_expected_shift)
df['shift_mismatch'] = df['rep_shift'] != df['expected_shift']

scheduled_df = df[df['demo_scheduled_flag']].copy()
no_show_df = scheduled_df[~scheduled_df['demo_joined_flag']]
mismatch_no_shows = no_show_df['shift_mismatch'].sum()
pct_no_shows_mismatch = (mismatch_no_shows / len(no_show_df)) * 100 if len(no_show_df) > 0 else 0

In [9]:
# 5. PRINT EXPLICIT CONSOLE REPORT
print("="*75)
print("              BRIGHTCHAMPS FUNNEL & FINANCIAL LEAK REPORT              ")
print("="*75)
print(f"Total Leads Analyzed (2 Months): {total_leads:,} | Monthly Average: {monthly_leads:,.0f}")
print("-" * 75)
print(f"Stage 1: Total Leads Created      : {total_leads:5,}  (100.0%) | {monthly_leads:6,.1f} / mo")
print(f"Stage 2: Demos Scheduled          : {demos_scheduled:5,}  ({(demos_scheduled/total_leads)*100:5.1f}%) | {monthly_scheduled:6,.1f} / mo")
print(f"Stage 3: Demos Joined             : {demos_joined:5,}  ({(demos_joined/total_leads)*100:5.1f}%) | {monthly_joined:6,.1f} / mo")
print(f"Stage 4: Demos Completed          : {demos_completed:5,}  ({(demos_completed/total_leads)*100:5.1f}%) | {demos_completed/num_months:6,.1f} / mo")
print(f"Stage 5: Converted (Paid Sales)   : {converted_customers:5,}  ({(converted_customers/total_leads)*100:5.1f}%) | {converted_customers/num_months:6,.1f} / mo")
print("-" * 75)

print("\n--- FINANCIAL LEAK ANALYSIS (STAGE 2 -> STAGE 3 DROP-OFF) ---")
print(f"• Monthly Scheduled Demos Failing to Join : {monthly_no_shows:,.1f} leads / month")
print(f"• Downstream Completion Rate (Joined->Done): {completion_rate*100:.1f}%")
print(f"• Downstream Close Rate (Done->Paid)       : {closing_rate*100:.1f}%")
print(f"• Expected Value of 1 Joined Demo         : ₹{expected_value_per_joined_demo:,.2f}")
print(f"---------------------------------------------------------------------------")
print(f"• MONTHLY UNREALIZED REVENUE LEAK          : ₹{monthly_unrealized_revenue:,.2f} (~₹{monthly_unrealized_revenue/100000:.2f} Lakhs)")
print(f"• MONTHLY SUNK MARKETING CAC LOST         : ₹{monthly_sunk_cac:,.2f} (~₹{monthly_sunk_cac/100000:.2f} Lakhs)")
print(f"• TOTAL COMBINED MONTHLY FINANCIAL LEAK    : ₹{monthly_unrealized_revenue + monthly_sunk_cac:,.2f}")
print("---------------------------------------------------------------------------")

print("\n--- ROOT CAUSE & SHIFT MISMATCH METRICS ---")
print(f"• Total Scheduled No-Shows (2 Months)      : {len(no_show_df):,}")
print(f"• No-Shows Linked to Timezone/Shift Mismatch: {mismatch_no_shows:,} ({pct_no_shows_mismatch:.1f}%)")
print("="*75)

              BRIGHTCHAMPS FUNNEL & FINANCIAL LEAK REPORT              
Total Leads Analyzed (2 Months): 5,000 | Monthly Average: 2,500
---------------------------------------------------------------------------
Stage 1: Total Leads Created      : 5,000  (100.0%) | 2,500.0 / mo
Stage 2: Demos Scheduled          : 3,229  ( 64.6%) | 1,614.5 / mo
Stage 3: Demos Joined             : 2,060  ( 41.2%) | 1,030.0 / mo
Stage 4: Demos Completed          : 1,786  ( 35.7%) |  893.0 / mo
Stage 5: Converted (Paid Sales)   :   362  (  7.2%) |  181.0 / mo
---------------------------------------------------------------------------

--- FINANCIAL LEAK ANALYSIS (STAGE 2 -> STAGE 3 DROP-OFF) ---
• Monthly Scheduled Demos Failing to Join : 584.5 leads / month
• Downstream Completion Rate (Joined->Done): 86.7%
• Downstream Close Rate (Done->Paid)       : 20.3%
• Expected Value of 1 Joined Demo         : ₹10,543.69
---------------------------------------------------------------------------
• MONTHLY UNREALIZE

In [11]:
# 6. FIXED AUTOMATION AGENT SIMULATION
def run_automation_agent_sample(dataframe, sample_size=3):
    print("\n================ RUNNING AUTOMATED AGENT PROTOTYPE ================")
    
    # Filter for scheduled leads so time_str is populated
    scheduled_leads = dataframe[dataframe['demo_scheduled_flag']].copy()
    sample_records = scheduled_leads.sample(sample_size, random_state=42).to_dict(orient='records')
    
    # Dynamic Shift Mapping Matrix
    tz_mapping = {
        'America/New_York': 'US_SHIFT',
        'America/Los_Angeles': 'US_SHIFT',
        'Europe/London': 'EMEA_SHIFT',
        'Asia/Dubai': 'EMEA_SHIFT',
        'Asia/Riyadh': 'EMEA_SHIFT',
        'Asia/Singapore': 'SEA_SHIFT',
        'Asia/Ho_Chi_Minh': 'SEA_SHIFT',
        'Australia/Sydney': 'SEA_SHIFT'
    }
    
    for idx, record in enumerate(sample_records, 1):
        lead_id = record.get('lead_id')
        tz = record.get('parent_timezone')
        curr_shift = record.get('rep_shift')
        
        # Determine recommended shift dynamically
        exp_shift = tz_mapping.get(tz, 'IST_SHIFT')
        
        # Corrected field name lookup: demo_scheduled_at
        demo_time = record.get('demo_scheduled_at')
        if pd.isna(demo_time):
            time_str = "To Be Scheduled"
        else:
            time_str = pd.to_datetime(demo_time).strftime('%Y-%m-%d %H:%M UTC')
        
        needs_reroute = (curr_shift != exp_shift)
        
        wa_payload = {
            "lead_id": lead_id,
            "timezone": tz,
            "assigned_shift": curr_shift,
            "recommended_shift": exp_shift,
            "status": "REASSIGN_SHIFT" if needs_reroute else "CONFIRMED",
            "whatsapp_message": f"Hi Parent! Your BrightChamps Demo is set for {time_str} ({tz}). Reply 1 to Confirm, 2 to Reschedule."
        }
        
        print(f"\n[Lead #{idx}: {lead_id}]")
        print(json.dumps(wa_payload, indent=2, default=str))

run_automation_agent_sample(df)


================ RUNNING AUTOMATED AGENT PROTOTYPE ================

[Lead #1: L102683]
{
  "lead_id": "L102683",
  "timezone": "Asia/Singapore",
  "assigned_shift": "IST_SHIFT",
  "recommended_shift": "SEA_SHIFT",
  "status": "REASSIGN_SHIFT",
  "whatsapp_message": "Hi Parent! Your BrightChamps Demo is set for 2026-06-06 04:07 UTC (Asia/Singapore). Reply 1 to Confirm, 2 to Reschedule."
}

[Lead #2: L102695]
{
  "lead_id": "L102695",
  "timezone": "America/New_York",
  "assigned_shift": "US_SHIFT",
  "recommended_shift": "US_SHIFT",
  "status": "CONFIRMED",
  "whatsapp_message": "Hi Parent! Your BrightChamps Demo is set for 2026-06-23 03:23 UTC (America/New_York). Reply 1 to Confirm, 2 to Reschedule."
}

[Lead #3: L101062]
{
  "lead_id": "L101062",
  "timezone": "Australia/Sydney",
  "assigned_shift": "IST_SHIFT",
  "recommended_shift": "SEA_SHIFT",
  "status": "REASSIGN_SHIFT",
  "whatsapp_message": "Hi Parent! Your BrightChamps Demo is set for 2026-08-03 19:56 UTC (Australia/Sydney)